# P2 D-256 StyleGAN2 — Colab Training (Drive-based)

**Workflow:** `MyDrive/osai/` 에 repo 통째로 두고 작업. 코드와 ckpt는 Drive에 그대로, **데이터 zip만 Colab 로컬로 1회 cp** (Drive over FUSE는 random-access I/O가 매우 느려 학습 throughput에 치명적).

**Hardware:** A100 GPU (Runtime → Change runtime type → A100).

**Pre-flight:**
- `MyDrive/osai/` 가 이미 있어야 함. 없으면 첫 셀이 자동 clone.
- `MyDrive/osai/p2/train_50k_256.zip` 위치에 학습 데이터 두기 (이미 GitHub repo의 p2/ 디렉토리에 있으면 Drive로 한 번만 업로드).

In [ ]:
# 1. GPU check — must be A100 (or L4 if A100 unavailable)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Sync repo into MyDrive/osai (clone if absent, pull if present)
import os
WORK = '/content/drive/MyDrive/osai'
if not os.path.exists(WORK):
    !git clone https://github.com/geniemo/osai.git {WORK}
elif not os.path.exists(f'{WORK}/.git'):
    print(f'{WORK} exists but is not a git repo. Either delete it or clone elsewhere.')
else:
    %cd {WORK}
    !git fetch origin
    !git checkout improve
    !git pull --rebase origin improve
%cd {WORK}
!git rev-parse --short HEAD

In [ ]:
# 4. Install dependencies
!pip install -q pyyaml wandb pytorch-fid onnx onnxruntime scipy

In [ ]:
# 5. Copy training zip from Drive → Colab local disk (CRITICAL for throughput)
# Drive FUSE has terrible random-access perf; the ZipImageDataset reads a random
# entry per sample, so we MUST have the zip on Colab-local /content/ for training.
import os, shutil
src = '/content/drive/MyDrive/osai/p2/train_50k_256.zip'
dst_dir = '/content/p2_data'
dst = f'{dst_dir}/train_50k_256.zip'
os.makedirs(dst_dir, exist_ok=True)
if not os.path.exists(dst):
    print(f'Copying {src} → {dst}…')
    shutil.copy(src, dst)
print('Local zip:', os.path.getsize(dst) / 1e9, 'GB')

# Symlink to the config's expected path inside Drive repo
link = '/content/drive/MyDrive/osai/p2/data/train_50k_256.zip'
os.makedirs(os.path.dirname(link), exist_ok=True)
if os.path.islink(link) or os.path.exists(link):
    os.remove(link)
os.symlink(dst, link)
!ls -la /content/drive/MyDrive/osai/p2/data/

In [ ]:
# 6. WandB login
import wandb
wandb.login()

## Launch — first session

Working dir is `MyDrive/osai/`. ckpts will land in `MyDrive/osai/p2/runs/d256_main/` (Drive) so they survive Colab disconnects automatically — no separate backup needed.

In [ ]:
%cd /content/drive/MyDrive/osai
!PYTHONPATH=. python p2/train.py --config p2/configs/d256.yaml 2>&1 | tee -a p2/runs/d256_main/train.log

## Resume — after disconnect

Re-run cells 2 (mount), 3 (git pull), 4 (pip), 5 (data copy if Colab-local was wiped), 6 (wandb login). Then this cell auto-finds the latest ckpt in Drive:

In [ ]:
import glob
%cd /content/drive/MyDrive/osai
ckpts = sorted(glob.glob('p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Latest ckpt:', latest)
if latest:
    !PYTHONPATH=. python p2/train.py --config p2/configs/d256.yaml --resume {latest} 2>&1 | tee -a p2/runs/d256_main/train.log

## Self-measure FID (between sessions)

Optional but recommended. Requires `MyDrive/osai/p2/valid_10k_256.zip`. Real-stats cached once.

In [ ]:
# One-time: copy valid zip locally, extract, cache real-stats (Drive write OK — done once)
import os, shutil
src = '/content/drive/MyDrive/osai/p2/valid_10k_256.zip'
dst = '/content/p2_data/valid_10k_256.zip'
if not os.path.exists(dst):
    shutil.copy(src, dst)
valid_dir = '/content/p2_data/valid_10k_256_dir'
if not os.path.isdir(valid_dir):
    os.makedirs(valid_dir, exist_ok=True)
    !cd {valid_dir} && unzip -q -o {dst}
%cd /content/drive/MyDrive/osai
!python -m pytorch_fid {valid_dir} --save-stats p2/checkpoints/fid_stats_256.npz

In [ ]:
# Measure FID on latest ckpt (or specify path)
import glob
%cd /content/drive/MyDrive/osai
ckpts = sorted(glob.glob('p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Evaluating:', latest)
if latest:
    !PYTHONPATH=. python p2/eval_fid.py --ckpt {latest} --stats p2/checkpoints/fid_stats_256.npz --n 8000 --batch 32

## ONNX export — for leaderboard

In [ ]:
%cd /content/drive/MyDrive/osai
!PYTHONPATH=. python p2/export_onnx.py \
    --ckpt p2/runs/d256_main/final.pt \
    --out p2/checkpoints/model.onnx
# model.onnx is now in MyDrive/osai/p2/checkpoints/ — download from Drive UI